In [1]:
from ax.service.ax_client import AxClient
import pandas as pd
import os
import torch
import numpy as np
from ax.utils.notebook.plotting import render, init_notebook_plotting


# Modern Analysis API for Parallel Coordinates
from ax.analysis.plotly.parallel_coordinates import ParallelCoordinatesPlot
import plotly.graph_objects as go
import itertools 




[WARNING 02-04 01:09:26] ax.service.utils.with_db_settings_base: Ax currently requires a sqlalchemy version below 2.0. This will be addressed in a future release. Disabling SQL storage in Ax for now, if you would like to use SQL storage please install Ax with mysql extras via `pip install ax-platform[mysql]`.


In [6]:
snapshot_path = "/home/leozhu/CaloQuVAE/wandb-outputs/BO_2026-02-01_06-03-10/experiment_snapshot.json"

# 2. Load the client
try:
    restored_client = AxClient.load_from_json_file(filepath=snapshot_path)
    print("Snapshot loaded successfully.")
except Exception as e:
    print(f"Error loading snapshot: {e}")

# 3. View all trials as a Pandas DataFrame
# This is the best way to see the status of every trial
df = restored_client.get_trials_data_frame()

# Display key columns
# 'trial_status' will tell you if a trial was COMPLETED, RUNNING, or FAILED
columns_of_interest = ["trial_index", "trial_status", "loss_metric"]
print(df[columns_of_interest][:32])

# 4. Check specifically for 'RUNNING' trials
# These are the ones that were active during the crash (and are likely lost/need to be reset)
running_trials = df[df["trial_status"] == "RUNNING"]
if not running_trials.empty:
    print(f"\nThere are {len(running_trials)} trials marked as RUNNING.")
    print("These correspond to the processes that were killed.")
else:
    print("\nNo trials are currently marked as RUNNING.")

# 5. See the best parameters found so far (from completed trials)
if any(df["trial_status"] == "COMPLETED"):
    best_params, best_values = restored_client.get_best_parameters()
    print("\nBest Parameters found so far:")
    print(best_params)
    print(f"Best Loss: {best_values}")

Snapshot loaded successfully.
    trial_index trial_status  loss_metric
0             0    COMPLETED    11.991284
1             1       FAILED          NaN
2             2    COMPLETED    49.088014
3             3    COMPLETED    63.141223
4             4    COMPLETED    66.323423
5             5    COMPLETED    64.230030
6             6    COMPLETED    59.849267
7             7    COMPLETED    64.712443
8             8       FAILED          NaN
9             9       FAILED          NaN
10           10    COMPLETED    45.978246
11           11    COMPLETED    34.268617
12           12    COMPLETED    44.305555
13           13    COMPLETED    61.245404
14           14    COMPLETED    59.175767
15           15    COMPLETED    68.390521
16           16    COMPLETED    37.520055
17           17    COMPLETED    11.209551
18           18    COMPLETED    44.806478
19           19    COMPLETED    13.843658
20           20    COMPLETED    14.757896
21           21    COMPLETED    14.386004
22  

In [3]:
dead_trials = [21, 22]

print(f"Marking trials {dead_trials} as ABANDONED...")

# 3. Mark them as ABANDONED
# "ABANDONED" tells Ax: "Don't use this data, and don't count it towards the total trial budget."
# Ax will likely re-suggest similar parameters later if the algorithm thinks they were promising.
for trial_index in dead_trials:
    restored_client.abandon_trial(trial_index=trial_index)

# 4. Save the clean state back to the file
restored_client.save_to_json_file(filepath=snapshot_path)

print("Snapshot updated. You can now restart your training script.")

In [4]:

# --- CONFIGURATION ---
SNAPSHOT_PATH = "/home/leozhu/CaloQuVAE/wandb-outputs/BO_2026-02-01_06-03-10/experiment_snapshot.json"
OUTPUT_DIR = os.path.dirname(SNAPSHOT_PATH)

def recover_plots(snapshot_path, output_dir):
    if not os.path.exists(snapshot_path):
        logger.error(f"Snapshot not found at: {snapshot_path}")
        return

    logger.info(f"Loading snapshot from: {snapshot_path}")
    ax_client = AxClient.load_from_json_file(filepath=snapshot_path)
    experiment = ax_client.experiment
    metric_name = "loss_metric" 

    # --- 1. RECOVER OPTIMIZATION TRACE ---
    logger.info("Generating Optimization Trace...")
    try:
        # Returns an AxPlotConfig object
        trace_config = ax_client.get_optimization_trace(
        )
        
        # FIX: Extract data from AxPlotConfig and create a standard Plotly Figure
        # AxPlotConfig.data is a dictionary suitable for go.Figure
        trace_fig = go.Figure(trace_config.data)
        
        # Save
        trace_fig.write_html(os.path.join(output_dir, "recovered_optimization_trace.html"))
        logger.info("Saved trace plot.")

    except Exception as e:
        logger.error(f"Failed to recover trace plot: {e}", exc_info=True)

    logger.info("Generating Contour Plots...")
    try:
        # Refit model (required for predictions)
        logger.info("Refitting model...")
        ax_client.fit_model()

        # Get all tunable parameter names from the experiment
        param_names = list(experiment.search_space.parameters.keys())

        # Check if we have at least 2 parameters to plot
        if len(param_names) < 2:
            logger.warning("Need at least 2 parameters to create contour plots.")
        else:
            # Loop through every unique pair of parameters
            for param_x, param_y in itertools.combinations(param_names, 2):
                logger.info(f"Plotting {param_x} vs {param_y}...")
                
                try:
                    contour_config = ax_client.get_contour_plot(
                        param_x=param_x,
                        param_y=param_y,
                        metric_name=metric_name,
                    )
                    
                    # Convert to Plotly Figure
                    contour_fig = go.Figure(contour_config.data)
                    
                    # Create a filename safe string
                    filename = f"contour_{param_x}_vs_{param_y}.html"
                    contour_fig.write_html(os.path.join(output_dir, filename))
                    
                except Exception as inner_e:
                    # Catch errors for specific pairs (e.g. if a param is constant)
                    logger.warning(f"Could not plot {param_x} vs {param_y}: {inner_e}")

            logger.info("Saved all contour plots.")

    except Exception as e:
        logger.error(f"Failed to recover contour plots: {e}", exc_info=True)    # --- 3. RECOVER PARALLEL COORDINATES ---
    logger.info("Generating Parallel Coordinates...")
    try:
        analysis = ParallelCoordinatesPlot()
        
        # FIX: .compute() returns a LIST of AnalysisCards
        cards = analysis.compute(
            experiment=experiment,
            generation_strategy=ax_client.generation_strategy
        )
        
        if cards:
            # Get the first card and extract the figure using .get_figure()
            parallel_fig = cards.get_figure()
            
            parallel_fig.update_layout(title="Parallel Coordinates (Recovered)")
            parallel_fig.write_html(os.path.join(output_dir, "recovered_parallel_coordinates.html"))
            logger.info("Saved parallel coordinates.")
        else:
            logger.warning("No parallel coordinates cards returned.")
        
    except Exception as e:
        logger.error(f"Failed to recover parallel coordinates: {e}", exc_info=True)

    logger.info(f"Recovery complete. Outputs saved to {output_dir}")

recover_plots(SNAPSHOT_PATH, OUTPUT_DIR)


NameError: name 'logger' is not defined